# Stage 4 Baseline Runner (Colab)

This notebook launches the stage 4 baseline training pipeline from the repository and stores all training artifacts in Google Drive.

What it does:
- mounts Google Drive;
- clones the GitHub repository and checks out the working branch;
- installs the required Python packages;
- runs the baseline training script against the dataset in Google Drive;
- saves weights, plots, metrics, and a report draft to `stage4_outputs`.


## How to use

1. Open this notebook in Google Colab.
2. Enable GPU: `Runtime -> Change runtime type -> T4 GPU` (or any available GPU).
3. Run the cells from top to bottom.
4. If needed, change `GIT_BRANCH` or `PROJECT_ROOT` in the config cell.
5. After training, open `stage4_outputs` in Google Drive and use the saved artifacts for the stage 4 report.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path

GITHUB_REPO_URL = 'https://github.com/MasterKSAF/MyProject1.git'
GIT_BRANCH = 'codex/stage3-dataset-prep'
REPO_DIR = Path('/content/MyProject1_repo')

PROJECT_ROOT = Path('/content/drive/MyDrive/MyProject1')
MANIFEST_DIR = PROJECT_ROOT / 'stage3_outputs_colab'
DRIVE_DATASET_ROOT = PROJECT_ROOT / 'DB'
LOCAL_RUNTIME_ROOT = Path('/content/MyProject1_runtime')
LOCAL_DATASET_ROOT = LOCAL_RUNTIME_ROOT / 'DB'

IMAGE_SIZE = 160
BATCH_SIZE = 64
NUM_EPOCHS = 5
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 4
RANDOM_SEED = 42
USE_PRETRAINED = True

print('PROJECT_ROOT      =', PROJECT_ROOT)
print('MANIFEST_DIR      =', MANIFEST_DIR)
print('DRIVE_DATASET_ROOT=', DRIVE_DATASET_ROOT)
print('LOCAL_DATASET_ROOT=', LOCAL_DATASET_ROOT)
print('REPO_DIR          =', REPO_DIR)


In [ ]:
assert PROJECT_ROOT.exists(), f'Project root not found: {PROJECT_ROOT}'
assert DRIVE_DATASET_ROOT.exists(), f'Dataset folder not found: {DRIVE_DATASET_ROOT}'
assert MANIFEST_DIR.exists(), f'Manifest folder not found: {MANIFEST_DIR}'
assert (MANIFEST_DIR / 'train_manifest.csv').exists(), 'train_manifest.csv is missing'
assert (MANIFEST_DIR / 'val_manifest.csv').exists(), 'val_manifest.csv is missing'
assert (MANIFEST_DIR / 'test_manifest.csv').exists(), 'test_manifest.csv is missing'

print('Google Drive data is ready.')


In [ ]:
import shutil

%cd /content
LOCAL_RUNTIME_ROOT.mkdir(parents=True, exist_ok=True)
if LOCAL_DATASET_ROOT.exists():
    shutil.rmtree(LOCAL_DATASET_ROOT)
shutil.copytree(DRIVE_DATASET_ROOT, LOCAL_DATASET_ROOT)
print('Dataset copied to local runtime:', LOCAL_DATASET_ROOT)


In [ ]:
import shutil
import subprocess

%cd /content
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run(['git', 'clone', GITHUB_REPO_URL, str(REPO_DIR)], check=True)
%cd /content/MyProject1_repo
subprocess.run(['git', 'checkout', GIT_BRANCH], check=True)
subprocess.run(['git', 'status', '--short', '--branch'], check=True)


In [ ]:
%cd /content/MyProject1_repo
import subprocess
import sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-stage4.txt'], check=True)


In [ ]:
%cd /content/MyProject1_repo

pretrained_flag = '' if USE_PRETRAINED else '--no-pretrained'

!python -u scripts/stage4_baseline_train.py \
  --project-root "$PROJECT_ROOT" \
  --dataset-root "$LOCAL_DATASET_ROOT" \
  --manifest-dir "$MANIFEST_DIR" \
  --image-size "$IMAGE_SIZE" \
  --batch-size "$BATCH_SIZE" \
  --num-epochs "$NUM_EPOCHS" \
  --learning-rate "$LEARNING_RATE" \
  --weight-decay "$WEIGHT_DECAY" \
  --num-workers "$NUM_WORKERS" \
  --random-seed "$RANDOM_SEED" \
  $pretrained_flag


In [ ]:
import json
from IPython.display import Image, display

summary_path = PROJECT_ROOT / 'stage4_outputs' / 'reports' / 'run_summary.json'
metrics_path = PROJECT_ROOT / 'stage4_outputs' / 'reports' / 'test_metrics.json'
history_plot_path = PROJECT_ROOT / 'stage4_outputs' / 'plots' / 'training_history.png'
confusion_matrix_path = PROJECT_ROOT / 'stage4_outputs' / 'plots' / 'confusion_matrix.png'

required_paths = [summary_path, metrics_path, history_plot_path, confusion_matrix_path]
missing_paths = [path for path in required_paths if not path.exists()]
if missing_paths:
    missing_preview = '\n'.join(str(path) for path in missing_paths)
    raise FileNotFoundError(
        'Training artifacts are not ready yet. Run the training cell successfully first.\n'
        f'Missing files:\n{missing_preview}'
    )

with open(summary_path, 'r', encoding='utf-8') as fp:
    summary = json.load(fp)

with open(metrics_path, 'r', encoding='utf-8') as fp:
    metrics = json.load(fp)

print('Device        :', summary['device'])
print('Model         :', summary['model_name'])
print('Test accuracy :', round(metrics['test_accuracy'], 4))
print('Macro F1      :', round(metrics['macro_avg_f1'], 4))
print('Output dir    :', summary['output_dir'])

display(Image(filename=str(history_plot_path)))
display(Image(filename=str(confusion_matrix_path)))


## Saved artifacts

After the run, these files will be available in Google Drive inside `MyProject1/stage4_outputs`:
- `models/resnet18_best_model.pth`
- `plots/training_history.png`
- `plots/confusion_matrix.png`
- `reports/history.csv`
- `reports/test_metrics.json`
- `reports/test_predictions.csv`
- `reports/stage4_report_draft.md`
- `reports/run_summary.json`

These artifacts are enough to start writing the Word report for stage 4.
